In [1]:
import json
import re
import time
import requests

# 读取 JS
with open("places_full.js", "r", encoding="utf-8") as f:
    content = f.read()

json_str = re.sub(r"export const places\s*=\s*", "", content)
json_str = json_str.strip().rstrip(";")

places = json.loads(json_str)

In [10]:
def get_postcode(lat, lon):
    url = f"https://api.postcodes.io/postcodes?lon={lon}&lat={lat}"

    try:
        res = requests.get(url)
        data = res.json()

        if data["status"] == 200 and data["result"]:
            return data["result"][0]["postcode"]

    except Exception as e:
        print("Error:", e)

    return None

In [9]:
import time

seen_coords = {}



for i, p in enumerate(places):

    lat = p.get("latitude")
    lon = p.get("longitude")

    if lat is None or lon is None:
        continue

    # ---------- 1️⃣ 补 postcode ----------
    existing = p.get("meta", {}).get("card_postcode")

    if not existing:

        key = f"{lat},{lon}"

        if key in seen_coords:
            postcode = seen_coords[key]
        else:
            postcode = get_postcode(lat, lon)
            seen_coords[key] = postcode
            time.sleep(0.2)   # 比 nominatim 快很多

        if "meta" not in p:
            p["meta"] = {}

        p["meta"]["card_postcode"] = postcode

        print(i, postcode)

    # ---------- 2️⃣ 删除重复字段 ----------
    p.pop("location", None)
    p.pop("name", None)
    p.pop("type", None)

Error: name 'requests' is not defined
0 None
Error: name 'requests' is not defined
1 None
Error: name 'requests' is not defined
2 None
Error: name 'requests' is not defined
3 None
Error: name 'requests' is not defined
4 None
Error: name 'requests' is not defined
7 None
Error: name 'requests' is not defined
8 None
Error: name 'requests' is not defined
9 None
Error: name 'requests' is not defined
10 None
Error: name 'requests' is not defined
11 None
Error: name 'requests' is not defined
14 None
Error: name 'requests' is not defined
15 None
Error: name 'requests' is not defined
16 None
Error: name 'requests' is not defined
17 None
Error: name 'requests' is not defined
18 None
Error: name 'requests' is not defined
19 None
Error: name 'requests' is not defined
22 None
Error: name 'requests' is not defined
23 None
Error: name 'requests' is not defined
25 None
Error: name 'requests' is not defined
26 None
Error: name 'requests' is not defined
29 None
Error: name 'requests' is not defined
30 N

In [ ]:
with open("places_full_updated.js", "w", encoding="utf-8") as f:
    f.write("export const places = ")
    json.dump(places, f, indent=2)
    f.write(";")